In [ ]:
pip install nltk beautifulsoup4 emoji

In [ ]:
import pandas as pd
import re
import nltk
import emoji
from bs4 import BeautifulSoup

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")

from bs4 import MarkupResemblesLocatorWarning
import warnings

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [ ]:
def reduce_repeated_chars(word):
    # FIX: Reduce to 1 char (not 2) — "fuuuu" -> "fu" has no semantic value;
    # reducing to 1 gives cleaner tokens for NLP models
    return re.sub(r'(.)\1{2,}', r'\1', word)

In [ ]:
def normalize_profanity(text):
    text = re.sub(r"f\*+k", "fuck", text)
    text = re.sub(r"s\*+t", "shit", text)
    text = re.sub(r"b\*+ch", "bitch", text)
    return text

In [ ]:
def clean_text_advanced(text):

    # Lowercase
    text = text.lower()

    # Remove HTML
    text = BeautifulSoup(text, "html.parser").get_text()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove @mentions
    text = re.sub(r"@\w+", "", text)

    # Remove hashtags symbol
    text = re.sub(r"#", "", text)

    # FIX: Add spaces around emoji descriptions so they tokenize as separate words
    # e.g. "hello🤬world" -> "hello :face_with_symbols_on_mouth: world" -> "hello face with symbols on mouth world"
    text = emoji.demojize(text, delimiters=(" :", ": "))

    # Normalize censored profanity
    text = normalize_profanity(text)

    # Remove special characters (keep only letters and spaces)
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # Reduce repeated characters
    words = text.split()
    words = [reduce_repeated_chars(w) for w in words]

    # Remove stopwords
    words = [w for w in words if w not in stop_words]

    # Lemmatization
    words = [lemmatizer.lemmatize(w) for w in words]

    # FIX: Remove empty strings that may result from cleaning
    words = [w for w in words if w.strip()]

    text = " ".join(words)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
datasets = [
    "dataset1.csv",
    "dataset2.csv",
    "dataset3.csv",
    "dataset4.csv",
    "dataset5.csv"
]

processed_datasets = []

for file in datasets:
    print("Processing:", file)
    df = pd.read_csv(file)
    df["clean_comment"] = df["comment_text"].apply(clean_text_advanced)
    processed_datasets.append(df)
    df.to_csv(f"processed_{file}", index=False)  

In [ ]:
df = pd.read_csv("processed_dataset1.csv")

df.head()

In [ ]:
# Verify preprocessing is working: compare raw vs cleaned side by side
df[["comment_text", "clean_comment"]].head(5)

In [ ]:
# Optional: Check label distribution to spot class imbalance
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
print(df[label_cols].sum().sort_values(ascending=False))